In [3]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from sklearn.model_selection import train_test_split

# =========================================================
# CONFIG GERAL
# =========================================================
INPUT_PATH = Path("/Users/jose-cleiton/dev/script_pncp/data/dataset/pncp_exemplos_2000.csv")
OUTPUT_DIR = Path("/Users/jose-cleiton/dev/script_pncp/data/dataset/preparado_bart_llm")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CLASSIFICATION_PATH = OUTPUT_DIR / "classificacao_llm_raw.csv"
FULL_PATH = OUTPUT_DIR / "dataset_bart_full.csv"
TRAIN_PATH = OUTPUT_DIR / "train.csv"
VAL_PATH = OUTPUT_DIR / "validation.csv"

TEXT_COL_CANDIDATES = ["texto", "text", "Objeto"]
VAL_SIZE = 0.2
SEED = 42

# tamanho do lote enviado ao modelo
BATCH_SIZE = 20

# atraso entre chamadas para reduzir risco de rate limit
SLEEP_SECONDS = 1.0

# =========================================================
# CONFIG DO PROVEDOR
# =========================================================
# "openai" ou "gemini"
PROVIDER = "gemini"

PROVIDERS = {
    "openai": {
        "base_url": None,
        "api_key": os.environ.get("OPENAI_API_KEY"),
        "default_model": "gpt-5.4-mini",
    },
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key": os.environ.get("GEMINI_API_KEY"),
        "default_model": "gemini-2.0-flash",
    },
}

cfg = PROVIDERS[PROVIDER]

if not cfg["api_key"]:
    raise ValueError(
        f"API key não encontrada para o provedor '{PROVIDER}'. "
        f"Verifique a variável de ambiente correspondente."
    )

client = OpenAI(
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
)

DEFAULT_MODEL = cfg["default_model"]

print(f"Provedor ativo: {PROVIDER} | Modelo padrão: {DEFAULT_MODEL}")

# =========================================================
# PROMPT DO SISTEMA
# =========================================================
SYSTEM_PROMPT = """
Você é um classificador de itens de licitação e compras públicas.

Objetivo:
Dizer se o item serve para o meu negócio.

Considere como "serve_para_mim = true" itens relacionados a:
- software de controle de acesso
- software de segurança
- software de monitoramento
- software para controle de portaria
- software para registro de frequência
- software de gestão, quando ligado a acesso, ponto, portaria, facial ou monitoramento predial
- relógio de ponto
- cartão de ponto
- controle de ponto
- reconhecimento facial
- registrador eletrônico de ponto biométrico
- gerenciamento eletrônico de frequência
- controle de frequência
- registrador de ponto facial
- sistema com aplicativo facial
- controle de frequência facial
- controle de faces
- equipamento com biometria de face
- identificador facial
- bastão de ronda
- relógio protocolador
- relógio cartográfico
- bobina térmica para relógio de ponto
- papel termosensível para relógio de ponto
- controlador de acesso
- fechadura facial
- fechadura biométrica
- fechadura eletrônica facial
- fechadura eletromagnética
- controladora de acesso facial
- HID
- Hikvision
- detector de metais
- porta giratória
- portal detector de metal
- porta automática
- motor de portão
- catracas faciais
- catraca balcão
- catraca mecânica
- catracas eletrônicas
- catraca pedestal bidirecional
- catraca PNE
- controle de acesso
- controle de acesso facial
- cancela automática
- cancela eletrônica
- leitores faciais
- torniquete
- conjunto controle acesso área restrita
- iDFace
- botoeira

Marque como false quando o item estiver fora desse domínio, mesmo que use palavras parecidas.

Exemplos de fora do escopo:
- medicamentos
- merenda escolar
- cadeiras
- frota
- GPS
- telemetria
- software de multas
- prontuário eletrônico
- estação de solda
- buffet

Regras:
- Seja conservador.
- Se estiver ambíguo, use serve_para_mim = false, mas explique a ambiguidade no motivo.
- Responda somente no formato solicitado.
"""

# =========================================================
# SCHEMA JSON
# =========================================================
SCHEMA = {
    "type": "object",
    "properties": {
        "resultados": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "texto": {"type": "string"},
                    "serve_para_mim": {"type": "boolean"},
                    "categoria": {"type": "string"},
                    "motivo": {"type": "string"},
                    "confianca": {
                        "type": "string",
                        "enum": ["alta", "media", "baixa"]
                    },
                },
                "required": ["texto", "serve_para_mim", "categoria", "motivo", "confianca"],
                "additionalProperties": False,
            }
        }
    },
    "required": ["resultados"],
    "additionalProperties": False,
}

# =========================================================
# HELPERS
# =========================================================
def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Nenhuma das colunas esperadas foi encontrada: {candidates}")


def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def bool_to_label(value):
    if value is True:
        return 1
    if value is False:
        return 0
    return None


def batchify(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def classificar_itens(textos, model=None):
    model = model or DEFAULT_MODEL

    prompt_usuario = {
        "instrucoes": [
            "Classifique cada item e diga se serve para o meu negócio ou não.",
            "Retorne exatamente um resultado por texto enviado.",
            "Mantenha o campo 'texto' igual ao texto recebido.",
        ],
        "itens": textos,
    }

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(prompt_usuario, ensure_ascii=False)},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "classificacao_itens_negocio",
                "schema": SCHEMA,
                "strict": True,
            },
        },
    )

    content = response.choices[0].message.content
    return json.loads(content)


# =========================================================
# LOAD
# =========================================================
df = pd.read_csv(INPUT_PATH)

text_col = find_column(df, TEXT_COL_CANDIDATES)

print("Coluna de texto:", text_col)
print("Linhas originais:", len(df))

df = df[[text_col]].copy()
df = df.rename(columns={text_col: "text"})
df["text"] = df["text"].apply(normalize_text)
df = df[df["text"] != ""].drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Linhas únicas com texto válido:", len(df))

# =========================================================
# CLASSIFICAÇÃO LLM EM LOTES
# =========================================================
todos_textos = df["text"].tolist()
resultados_finais = []

for idx, lote in enumerate(batchify(todos_textos, BATCH_SIZE), start=1):
    print(f"\nProcessando lote {idx} | tamanho={len(lote)}")
    try:
        saida = classificar_itens(lote)
        resultados = saida.get("resultados", [])

        if len(resultados) != len(lote):
            print(
                f"Atenção: lote {idx} retornou {len(resultados)} resultados "
                f"para {len(lote)} textos."
            )

        resultados_finais.extend(resultados)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        print(f"Erro no lote {idx}: {e}")
        # fallback mínimo: salva lote como erro para revisão posterior
        for texto in lote:
            resultados_finais.append({
                "texto": texto,
                "serve_para_mim": False,
                "categoria": "erro_classificacao",
                "motivo": f"Erro ao classificar lote: {e}",
                "confianca": "baixa",
            })

# =========================================================
# DATAFRAME RAW
# =========================================================
raw_df = pd.DataFrame(resultados_finais)

# garante colunas
expected_cols = ["texto", "serve_para_mim", "categoria", "motivo", "confianca"]
for col in expected_cols:
    if col not in raw_df.columns:
        raw_df[col] = None

raw_df = raw_df[expected_cols].copy()
raw_df["label"] = raw_df["serve_para_mim"].apply(bool_to_label)

# remove textos vazios
raw_df["texto"] = raw_df["texto"].apply(normalize_text)
raw_df = raw_df[raw_df["texto"] != ""].copy()

# remove duplicados por texto, mantendo a primeira classificação
raw_df = raw_df.drop_duplicates(subset=["texto"]).reset_index(drop=True)

raw_df.to_csv(RAW_CLASSIFICATION_PATH, index=False, encoding="utf-8-sig")

print("\nArquivo bruto gerado:")
print(RAW_CLASSIFICATION_PATH)

# =========================================================
# DATASET FINAL PARA BART
# =========================================================
dataset_df = raw_df[["texto", "label"]].copy()
dataset_df = dataset_df.rename(columns={"texto": "text"})
dataset_df = dataset_df[dataset_df["label"].isin([0, 1])].copy()
dataset_df["label"] = dataset_df["label"].astype(int)

# remove duplicados exatos
dataset_df = dataset_df.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("\nLinhas válidas para treino:", len(dataset_df))
print("\nDistribuição de classes:")
print(dataset_df["label"].value_counts(dropna=False))

dataset_df.to_csv(FULL_PATH, index=False, encoding="utf-8-sig")

# =========================================================
# SPLIT ESTRATIFICADO
# =========================================================
train_df, val_df = train_test_split(
    dataset_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=dataset_df["label"],
)

train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df.to_csv(TRAIN_PATH, index=False, encoding="utf-8-sig")
val_df.to_csv(VAL_PATH, index=False, encoding="utf-8-sig")

print("\nArquivos gerados:")
print(TRAIN_PATH)
print(VAL_PATH)
print(FULL_PATH)

print("\nAmostra treino:")
print(train_df.head(5).to_string(index=False))

print("\nAmostra validação:")
print(val_df.head(5).to_string(index=False))

Provedor ativo: gemini | Modelo padrão: gemini-2.0-flash
Coluna de texto: texto
Linhas originais: 2000
Linhas únicas com texto válido: 1699

Processando lote 1 | tamanho=20

Processando lote 2 | tamanho=20

Processando lote 2 | tamanho=20

Processando lote 3 | tamanho=20

Processando lote 3 | tamanho=20

Processando lote 4 | tamanho=20

Processando lote 4 | tamanho=20

Processando lote 5 | tamanho=20

Processando lote 5 | tamanho=20

Processando lote 6 | tamanho=20

Processando lote 6 | tamanho=20

Processando lote 7 | tamanho=20

Processando lote 7 | tamanho=20

Processando lote 8 | tamanho=20

Processando lote 8 | tamanho=20

Processando lote 9 | tamanho=20

Processando lote 9 | tamanho=20

Processando lote 10 | tamanho=20

Processando lote 10 | tamanho=20

Processando lote 11 | tamanho=20

Processando lote 11 | tamanho=20

Processando lote 12 | tamanho=20

Processando lote 12 | tamanho=20

Processando lote 13 | tamanho=20

Processando lote 13 | tamanho=20

Processando lote 14 | tama